# Show1D

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/show1d.ipynb)

`Show1D` is the iteration dashboard for traces, optimizer losses, image line profiles, and reconstruction snapshots. It is useful when a ptychography or denoising run produces scalar metrics every iteration and object/probe images at lower-rate checkpoints.

This tutorial uses a compact synthetic ducky reconstruction so the notebook stays portable. The workflow is the same for real joint-time ptychography: compare loss traces, inspect linked reconstruction snapshots, hand selected snapshots to `Show2D`, and reopen a file-backed monitor after an overnight run.

In [ ]:
import pathlib
import subprocess
import sys
import tempfile

try:
    import google.colab  # noqa: F401
except Exception:
    pass
else:
    from google.colab import output

    output.enable_custom_widget_manager()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/bobleesj/quantem.widget.git"],
        check=True,
    )

import numpy as np

from quantem.widget import Show1D

## Compare loss traces with reconstruction snapshots

A joint reconstruction sweep often has many related loss curves: a frame-by-frame baseline plus several regularization strengths. `Show1D.from_loss_runs` accepts a mapping of labels to traces, then `snapshot(...)` attaches one logical image group to a chosen iteration or frame.

Each snapshot group below contains a reference image and multiple lambda candidates. The selected frame in the loss plot and the image grid stay linked.

In [ ]:
rng = np.random.default_rng(7)
frames = np.arange(12, dtype=np.float32)
lambdas = [0, 0.3, 1, 3, 10, 30, 100, 300]


def lambda_label(value):
    return "lambda_" + str(value).replace(".", "p")


def ducky_object(size=128, frame=0, lam=1, noise=0.02):
    grid = np.linspace(-1.0, 1.0, size, dtype=np.float32)
    row, col = np.meshgrid(grid, grid, indexing="ij")
    drift = 0.08 * np.sin(2 * np.pi * frame / max(1, len(frames) - 1))
    regularization = np.log10(float(lam) + 1.3)

    body = (((col + 0.16 + drift) / 0.55) ** 2 + ((row + 0.02) / 0.50) ** 2) < 1.0
    head = (((col - 0.30 + drift) / 0.27) ** 2 + ((row - 0.12) / 0.30) ** 2) < 1.0
    beak = np.exp(-((col - 0.62 + drift) ** 2 / 0.010 + (row - 0.12) ** 2 / 0.006))
    wing = np.exp(-((col + 0.10 + drift) ** 2 / 0.070 + (row + 0.10) ** 2 / 0.020))
    lattice = 0.08 * np.sin(42 * col + 0.6 * frame) * np.sin(38 * row)
    background = 0.10 * np.sin(8 * col + 0.4 * frame) + 0.08 * np.cos(7 * row)

    image = 0.20 * body + 0.28 * head + 0.16 * beak + 0.12 * wing + lattice + background
    blur_bias = 0.08 * abs(regularization - 1.1)
    image = image - blur_bias * (row**2 + col**2)
    image += noise * rng.standard_normal((size, size)).astype(np.float32)
    return image.astype(np.float32)


loss_runs = {}
for lam in lambdas:
    smooth = 111.5 + 1.2 / (1 + np.log10(float(lam) + 1.5)) + 0.45 * np.exp(-frames / 3.2)
    flicker = 0.10 * np.sin(0.9 * frames + 0.17 * float(lam))
    if lam in (0, 300):
        flicker += 0.20 * np.sin(1.7 * frames)
    loss_runs[f"lambda {lam:g}"] = smooth + flicker + rng.normal(0, 0.018, frames.size)

loss_runs["frame-by-frame"] = 112.9 + 0.08 * np.sin(frames)

widget = Show1D.from_loss_runs(
    loss_runs,
    x=frames,
    title="Ducky joint iterative ptychography: lambda sweep",
    x_label="frame",
    y_label="final loss",
    log_scale=False,
    snapshot_columns=4,
    snapshot_panel_width_px=720,
    snapshot_histogram_width=360,
    snapshot_histogram_height=52,
    sampling=0.20,
    units="nm",
    show_snapshot_profile=True,
    snapshot_profile_line=((38, 24), (92, 96)),
)

for frame in [0, 3, 5, 8, 11]:
    images = {"reference": ducky_object(frame=0, lam=3, noise=0.0)}
    for lam in lambdas:
        images[lambda_label(lam)] = ducky_object(frame=int(frame), lam=lam, noise=0.045)
    widget.snapshot(float(frame), label=f"frame {frame}", **images)

widget.goto_snapshot(2)
widget

## Open the selected snapshot group in Show2D

Use `to_show2d()` when a snapshot group needs deeper image inspection. The handoff preserves labels, colormap, scale-bar units, hidden/starred trial state, and linked contrast defaults.

In [ ]:
widget.to_show2d(
    group=2,
    images=["reference", "lambda_1", "lambda_10", "lambda_30"],
    title="Frame 5 reconstruction candidates",
)

## Profile a single image

`Show1D.from_image` samples a line profile from a 2D image and keeps the image context beside the trace. Coordinates use the scientific `(row, col)` convention.

In [ ]:
reference = ducky_object(frame=0, lam=3, noise=0.0)

Show1D.from_image(
    reference,
    line=((34, 22), (94, 100)),
    profile_width=3,
    sampling=0.20,
    x_unit="nm",
    title="Ducky object line profile",
    image_cmap="cividis",
)

## Reopen a file-backed live monitor

For overnight reconstructions, write one JSON object per line. Each event can contain scalar losses, metrics, warnings, review state, and snapshot paths relative to the monitor file. `Show1D.from_monitor_file(...)` rebuilds the dashboard after the run, while `Show1D.watch_run(..., refresh_s=5)` polls the same file during a live notebook session.

In [ ]:
run_dir = pathlib.Path(tempfile.mkdtemp(prefix="quantem_show1d_monitor_"))
snapshot_dir = run_dir / "snapshots"
snapshot_dir.mkdir(parents=True)
monitor_path = run_dir / "show1d_monitor.jsonl"

monitor_lambdas = [0.3, 1, 3, 10]
for i in range(12):
    losses = {}
    metrics = {}
    snapshots = {}
    for lam in monitor_lambdas:
        label = lambda_label(lam)
        loss = 112.0 + 0.9 * np.exp(-i / (2.0 + 0.3 * lam)) + 0.04 * np.sin(i + lam)
        losses[f"lambda {lam:g}"] = float(loss)
        metrics[label] = {
            "final_loss": float(loss),
            "flicker": float(0.04 + 0.015 * abs(np.log10(lam + 0.3) - 0.7)),
        }
        if i in (0, 4, 8, 11):
            image_path = snapshot_dir / f"{label}_i{i:03d}.npy"
            np.save(image_path, ducky_object(frame=i, lam=lam, noise=0.04))
            snapshots[label] = str(image_path.relative_to(run_dir))

    event = {"iteration": i, "losses": losses, "metrics": metrics}
    if snapshots:
        event["label"] = f"iter {i}"
        event["snapshots"] = snapshots
    if i == 8:
        event["warnings"] = ["lambda 0.3 has visible frame-to-frame flicker"]
    if i == 11:
        event["starred"] = ["lambda_3"]
        event["hidden"] = ["lambda_0p3"]
        event["notes"] = {"lambda_3": "best visual/loss balance"}
        event["tags"] = {"lambda_3": ["best lambda"]}

    Show1D.append_monitor_event(monitor_path, event)

monitor = Show1D.from_monitor_file(
    monitor_path,
    title="Reopened Show1D monitor",
    snapshot_columns=4,
    snapshot_panel_width_px=640,
    snapshot_histogram_width=320,
    show_review=False,
    log_scale=False,
)
monitor.goto_snapshot(3)
monitor

During a real run, display the polling widget once, then let the reconstruction keep appending JSONL events:

```python
live = Show1D.watch_run("/path/to/run/show1d_monitor.jsonl", refresh_s=5)
live
```

Use `live.stop_monitor()` when the notebook no longer needs to poll. If the notebook disconnects overnight, rerun `Show1D.from_monitor_file(...)` on the same file to rebuild the view from disk.